# Statistical inference

*Sampling variation, confidence intervals, and hypothesis tests*

With this chapter we begin Part III, the second stage of the course map: deciding whether a pattern we observe is genuine or due to random chance. In Part II we described the data, but from a description alone we cannot tell whether a pattern comes from the variables we are examining or from chance variation in the sample. In the spring of 2026, Prairie Wholesale offered free shipping to some of its online customers and made no change for the rest. The customers who received the offer placed larger orders on average. Is that the offer's effect, or luck? In this chapter we learn the tools for such questions: sampling distributions, confidence intervals, and hypothesis tests. We apply these tools to a decision Prairie Wholesale has to make: whether to extend the offer to every online customer.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-07-inference.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-07-inference.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly scipy
import sys
if "google.colab" in sys.modules:
    !git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

> **Tools in this chapter**
>
> | Tool | Why we use it here | Alternatives | Trade-off |
> |---|---|---|---|
> | scipy.stats | Classical statistical tests and distributions in one module | statsmodels (more detail per model), manual simulation | scipy has the standard tests in one line each; we build the simulated versions first so the one-liners are not black boxes |
>
> : {tbl-colwidths="[12,33,22,33]"}

## The pilot {#sec-ch7-intro}

Between March 2 and May 31, 2026, Prairie Wholesale conducted a test. Online customers were randomly split into two groups. The pilot group received free shipping on orders over \$200. For the control group, nothing changed. The company is interested in a single question: did free shipping increase order size?

The data of the earlier chapters was observational: customers acted on their own, with nothing assigned by the analyst. In [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html) we saw the limitation of such data: a third variable can explain any correlation, so a causal claim requires an experiment. In this case, the random split gives us an experiment with which to test the offer's effect: chance alone decided who received the offer, so apart from the offer itself, the two groups differ only by chance. No third variable, such as business type or customer size, can be systematically more common in one group than in the other. Whatever difference we observe can only have two possible explanations: the offer, or sampling luck. We spend the rest of the chapter distinguishing between the two.

We start by assembling the pilot data. The orders are online orders placed in the pilot window. The group labels are in the customer snapshot.

In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")

totals = orders.groupby("order_id").agg(
    total=("line_total", "sum"),
    date=("order_date", "first"),
    channel=("channel", "first"),
    customer_id=("customer_id", "first"),
)

window = totals[
    (totals["channel"] == "online")
    & (totals["date"] >= "2026-03-02")
    & (totals["date"] <= "2026-05-31")
].merge(snap[["customer_id", "pilot"]], on="customer_id").dropna(subset=["pilot"])

window.groupby("pilot")["total"].agg(["mean", "median", "count"]).round(2)

The pilot group's mean order is about \$120, and the control group's is about \$89. The difference is \$31 per order, a 35% lift. The sample sizes are 150 and 199 orders. Could two groups of a few hundred skewed order totals differ by \$31 by chance alone? Two means alone are not enough to answer that question. We need to know how much means of samples like these vary on their own.

## Sampling variation

A statistic calculated from a sample (a mean, a median, a difference) can change each time we select a different sample from the population. The extent to which the statistic changes from sample to sample can be measured. A difference between two group means, such as the pilot's \$31, is therefore convincing only if it is large compared to how much sample means vary on their own.

Because we have three years of orders to work with, sampling variation can be observed directly. For this example we use the complete order population; the same qualitative lesson applies to the pilot's more limited data. Take repeated random samples of 150 orders and calculate each sample's mean. For the random draws we use **NumPy**, the numerical array library that pandas itself is built on: `np.random.default_rng(0)` creates a random number generator. The argument 0 is a fixed seed, so your draws will match this book's.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
population = totals["total"].values

sample_means = [rng.choice(population, size=150).mean() for _ in range(2000)]
pd.Series(sample_means).describe().round(1)

> **Python note: the underscore loop variable**
>
> The comprehension `[... for _ in range(2000)]` repeats the draw 2,000 times and collects the results in a list. The underscore is the conventional Python name for a loop variable whose value is never used: each pass draws a fresh sample, and nothing in the expression needs the loop counter.

In [ ]:
px.histogram(pd.DataFrame({"sample_mean": sample_means}), x="sample_mean", nbins=60,
             labels={"sample_mean": "Mean of a 150-order sample ($)"})

This histogram shows two important facts. First, sample means differ: samples of 150 orders from the *same* population have means differing by tens of dollars. A difference between two sample means is therefore not meaningful in and of itself. Second, the distribution of sample means is roughly symmetric, despite individual orders being highly skewed ([Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html)). Averaging eliminates skew: as the sample size grows, the distribution of sample means approaches the **normal distribution**, the symmetric bell curve of basic statistics. This regularity is the **central limit theorem**, the basis of the standard formulas of statistics.

::: {.content-visible when-format="html"}
The explorer below draws samples from 800 order totals. Draw a hundred samples at n = 30, then increase n to 200 and draw again. You will see the distribution contract and move toward the normal distribution: both effects come from the larger sample size. Adding more draws at a fixed sample size does not narrow the distribution; the extra draws allow the distribution's shape to fill in.

<iframe src="../../assets/demos/sampling-simulator.html" width="100%" height="520" style="border:1px solid #d0d7de; border-radius:8px;" title="Sampling variation simulator"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an interactive simulator here: it repeatedly draws samples of adjustable size from the order population and accumulates the histogram of their means. The histogram contracts as n increases. @fig-sampling-dist shows the result for n = 150.
:::

## Confidence intervals by bootstrap

The simulation above relied on information we do not usually have: it sampled from a population we happened to have on hand. In actual inference we have one sample and cannot draw repeated samples from the population. In such cases, the **bootstrap** is the usual solution. The sample is treated as the best available proxy for the population. We resample *from the sample itself*, with replacement, many times. The spread of the resampled statistics estimates the sampling variation of the original statistic.

Let us bootstrap the control group's mean order value:

In [ ]:
control = window.loc[window["pilot"] == 0, "total"].values

rng = np.random.default_rng(1)     # a fresh, named generator per section
boot_means = [rng.choice(control, size=len(control)).mean() for _ in range(10_000)]
lo, hi = np.percentile(boot_means, [2.5, 97.5])
print(f"control mean ${control.mean():.2f}, 95% CI [${lo:.2f}, ${hi:.2f}]")

The interval is a **95% confidence interval**, a range of likely values for the true mean. The 95% refers to the method's track record: if we repeated this exercise on many fresh samples and computed an interval from each one in the same manner, about 95% of those intervals would contain the true mean. For the single interval in front of us, the practical reading follows: we act as though the true mean lies inside it, knowing that intervals built this way miss the truth about one time in twenty.

There is also a formula-based interval, the mean plus and minus roughly two standard errors, which here is almost identical to the bootstrap interval. We compute the formula version in one line with **SciPy**, the open-source scientific computing library of the Python ecosystem; its `stats` module contains the classical distributions and tests. We constructed the bootstrap first because every step of it is in plain sight. The formula is a fast approximation to that simulation.

In [ ]:
from scipy import stats

sem = stats.sem(control)                       # standard error of the mean
stats.t.interval(0.95, df=len(control) - 1, loc=control.mean(), scale=sem)

## Testing the pilot

We now return to the pilot and test it with a statistical method called **hypothesis testing**. The skeptic's view, called the **null hypothesis**, is that free shipping changed nothing and that the \$31 difference is sampling luck. The test is: if the null were true, how often would a difference this large appear by chance? That frequency is the **p-value**. A small p-value means the observed difference would be rare in a world where the free shipping offer had no effect on order sizes. That rarity is evidence against the null.

Because assignment was random, we can construct a test by hand. If the offer did nothing, the group labels are meaningless tags, so shuffling them should produce differences like the observed one. We therefore shuffle them, ten thousand times:

In [ ]:
observed = (window.loc[window["pilot"] == 1, "total"].mean()
            - window.loc[window["pilot"] == 0, "total"].mean())

values = window["total"].values
is_pilot = (window["pilot"] == 1).values

shuffled_diffs = []
for _ in range(10_000):
    labels = rng.permutation(is_pilot)         # shuffle who is "pilot"
    shuffled_diffs.append(values[labels].mean() - values[~labels].mean())
shuffled_diffs = np.array(shuffled_diffs)

p_value = (np.abs(shuffled_diffs) >= abs(observed)).mean()
print(f"observed diff ${observed:.2f}, permutation p-value {p_value:.4f}")

In [ ]:
fig = px.histogram(pd.DataFrame({"shuffled_diff": shuffled_diffs}), x="shuffled_diff",
                   nbins=80, labels={"shuffled_diff": "Difference under the null ($)"})
fig.add_vline(x=observed, line_color="#cf222e", annotation_text="observed",
              annotation_position="top right")
# shade the two tails at least as extreme as the observed difference
fig.add_vrect(x0=observed, x1=shuffled_diffs.max(), fillcolor="#cf222e", opacity=0.15, line_width=0)
fig.add_vrect(x0=shuffled_diffs.min(), x1=-observed, fillcolor="#cf222e", opacity=0.15, line_width=0)
fig.show()

This is a **permutation test**. In a world where free shipping did nothing, differences between the shuffled groups almost never reach the observed \$31. The p-value is about 0.0045: under the null, a difference this large appears about four or five times in a thousand experiments. The evidence against "the offer did not have an effect" is strong.

::: {.content-visible when-format="html"}
You can run this test yourself below, on the same 349 orders. You can see the p-value estimate stabilize as the shuffle count grows.

<iframe src="../../assets/demos/permutation-test.html" width="100%" height="430" style="border:1px solid #d0d7de; border-radius:8px;" title="permutation test"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an interactive permutation test here: the same 349 pilot orders, shuffle buttons, and a live p-value estimate that stabilizes as shuffles accumulate.
:::


The permutation test required custom code and ten thousand shuffles. However, like the confidence interval earlier, we can use a formula to replace the simulation: the **two-sample t-test**, the standard test for comparing two group means. We use Welch's version, which does not assume the groups have equal variances, an assumption rarely justified in order data:

In [ ]:
pilot_orders = window.loc[window["pilot"] == 1, "total"]
control_orders = window.loc[window["pilot"] == 0, "total"]

t_stat, p_t = stats.ttest_ind(pilot_orders, control_orders, equal_var=False)
print(f"Welch t = {t_stat:.2f}, p-value = {p_t:.4f}")

The t-test and the permutation test have p-values within half a thousandth of each other. The one-line test is a quick approximation of the simulation we constructed above.

## The size of the effect

A p-value concerns one narrow question: could a difference this large be chance alone? For the business, a second question remains: how large is the effect of free shipping on order sizes? We answer it with an estimate and a confidence interval, both computed with the bootstrap applied to the difference:

In [ ]:
a, b = pilot_orders.values, control_orders.values
boot_diffs = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean()
              for _ in range(10_000)]
lo, hi = np.percentile(boot_diffs, [2.5, 97.5])
print(f"lift per order ${observed:.2f}, 95% CI [${lo:.2f}, ${hi:.2f}]")

The interval is large: the data are consistent with a lift anywhere from roughly \$10 to about \$52 per order. Both facts belong in the report. The effect is different from chance (p ≈ 0.005). Its size is known only roughly (a factor of five between the ends of the interval). With more data the interval would be smaller, but in practical terms that would mean conducting a larger or longer experiment. The interval from a completed three-month pilot cannot be narrowed after the fact.

## Four rules for reading p-values

P-values are easy to misread. The four rules below concern the most consequential misreadings.

1. **The p-value is not the probability the null is true.** It is computed assuming the null: it measures how surprising the data would be in a hypothetical world where the offer had no effect. It carries no information about the probability that we are in that hypothetical world.
2. **Statistical significance and business significance are separate questions.** With enough orders, a \$0.30 lift becomes "significant". The confidence interval carries the business information; report it alongside every p-value.
3. **The threshold is a convention.** The traditional cutoff of 0.05 was chosen by custom, and the 95% confidence level from earlier in this chapter is the same convention seen from the other side: for the matching test, a 95% interval excludes zero exactly when the p-value is below 0.05. A p-value of 0.06 and a p-value of 0.04 are nearly identical evidence.
4. **One test, decided in advance.** Testing many differences and reporting the smallest p-value produces false significance from noise. Prairie Wholesale declared its one outcome of interest, the mean order total, before the pilot began, and committed to testing only that. When you must test many things, say so, and adjust expectations accordingly.

> **Do not outsource this**
>
> An AI assistant will write any test you name and narrate its output. Choosing *which* comparison answers the business question, and whether the design supports a causal reading, is the analyst's judgment. In the labs, the test code may be generated where permitted; the choice of test and the interpretation must be yours.


## Inverting a conditional probability: Bayes' rule

A **conditional probability** is the probability of one event given that another has occurred: for example, the probability that an invoice contains an error, given that a screening tool flagged it. The standard notation is $P(\text{error} \mid \text{flagged})$: $P$ stands for probability, and the vertical bar is read as "given". The first p-value rule concerned two conditional probabilities that are easy to confuse. The p-value is the probability of the observed data, given that the null is true. The misreading highlighted by the first rule runs in the reverse direction: the probability that the null is true, given the observed data. Moving from one to the other is called inverting the conditional probability. The inversion is a legitimate operation, but it requires an ingredient the p-value calculation does not contain: the base rate, meaning how common the thing in question is to begin with. For the pilot, the base rate would be how often offers like this one genuinely change behavior, a number we do not have. For the screening problem below, the base rate is how many invoices contain an error, a number the billing team knows. **Bayes' rule** is the arithmetic of this inversion. To see why the answer depends on the base rate, we work through a Prairie Wholesale screening problem.

The billing team runs an automated screening test for pricing mistakes on invoices. Three numbers describe the situation. Approximately 4% of invoices have an error (**the base rate**). The screening test detects 75% of erroneous invoices (**the sensitivity**). It also erroneously flags 9% of clean invoices (**the false-positive rate**). An invoice was just flagged. What is the probability that it has an error?

The intuitive response is near 75%, but the actual answer is about 26%. To see why, we set the formulas aside and count. We imagine a batch of 400 invoices, apply the three rates to it, and count how many invoices end up in each category:

In [ ]:
invoices = 400
erroneous = round(invoices * 0.04)          # 16 have errors
clean = invoices - erroneous                # 384 do not

caught = round(erroneous * 0.75)            # 12 erroneous invoices flagged
false_alarms = round(clean * 0.09)          # 35 clean invoices flagged

posterior = caught / (caught + false_alarms)
print(f"flagged invoices: {caught + false_alarms}, "
      f"actually erroneous: {caught}, P(error | flagged) = {posterior:.0%}")

The flagged group has 47 invoices, and only 12 of them contain an error: 12 divided by 47 is about 26%. The base rate explains the low number. Clean invoices outnumber erroneous ones 24 to 1, so a 9% false-alarm rate applied to the large clean group produces more flags (35) than a 75% catch rate applied to the small erroneous group (12). Bayes' rule states this counting in words: the probability of an error given a flag equals the count of flagged-and-erroneous invoices divided by the count of all flagged invoices, the same division we just performed. In the notation above, $P(\text{error} \mid \text{flagged}) = 12/47$, or about 26%. We counted first because the formula contains no idea beyond the counting.

::: {.content-visible when-format="html"}
The explorer below visualizes the counting argument: 400 cases, one square each, with the three rates on sliders. If you lower the base rate, you will see the flagged group turn mostly amber while the screening test's own accuracy never changes. Notice also that at low base rates the sensitivity slider has little effect on the question we are actually asking: whether the test catches 75% or 95% of true errors, a flag still rarely means an error, because the flagged group is dominated by false alarms from the much larger clean group.

<iframe src="../../assets/demos/bayes-grid.html" width="100%" height="560" style="border:1px solid #d0d7de; border-radius:8px;" title="Bayes rule population grid"></iframe>

The animation below shows the same recount as a pie chart. The full pie holds all 400 invoices; the prior probability of an error, 4%, is the share of the error slices. When you condition on the flag, the unflagged slices are removed and the two remaining slices are recounted out of the 47 flagged invoices, so the same blue slice now reads as 26%.

<iframe src="../../assets/demos/bayes-pies.html" width="100%" height="430" style="border:1px solid #d0d7de; border-radius:8px;" title="Bayes rule as a recount"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has two interactive figures here. The first is a population grid: 400 cases colored as caught, missed, false alarm, or cleared, with sliders for the base rate, sensitivity, and false-positive rate, and the posterior recomputed as a count of squares. The second is a pie chart of the same 400 cases, in which conditioning on the flag removes the unflagged slices and renormalizes the rest, moving the error share from the 4% prior to the 26% posterior.
:::

We can connect this idea to the course in two places. The first is backward, to the p-value rules. $P(\text{data} \mid \text{no effect}) = 0.005$ cannot become $P(\text{no effect} \mid \text{data}) = 0.005$ for the same reason a flag from the invoice screening test does not mean a 75% chance of error: the inversion requires the base rate. The second is forward, to prediction. In [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) we fit a churn model whose flags follow the same arithmetic. The **precision** of that model is this section's posterior by another name. When churn is rare, most of the customers a good model flags will not in fact churn, for the same reason most flagged invoices are clean. As such, budgeting a retention call for every flagged customer would result in wasted resources on calls to customers who were not going to leave.

## Evaluation: robustness of the finding

Our finding so far is that free shipping increased order values. To check the finding's robustness, we rerun the comparison under alternative reasonable analyses.

|  |  |
|---|---|
| **Metric** | does the conclusion (a positive effect, different from zero) hold under alternative reasonable analyses? |
| **Test** | three variants. |
| **Baseline** | the main analysis (the Welch t-test). |

: {tbl-colwidths="[18,82]"}

Two of the variants use tests this book does not otherwise teach: a t-test on log-scale totals, which shrinks the influence of the skew, and the **Mann-Whitney test**, a classical test that compares the two groups through ranks instead of means and therefore assumes nothing about the distribution's shape.

In [ ]:
checks = {
    "primary: Welch t on means": stats.ttest_ind(a, b, equal_var=False).pvalue,
    "permutation test": p_value,
    "log-scale totals (reduces skew)": stats.ttest_ind(np.log(a), np.log(b),
                                                     equal_var=False).pvalue,
    "Mann-Whitney (no normality assumption)": stats.mannwhitneyu(a, b).pvalue,
}
pd.Series(checks).round(4).to_frame("p-value")

The conclusion is the same under all four analyses, with p-values between roughly 0.0003 and 0.005. We report it to the leadership team with all four rows attached.

## The decision this informs

The pilot's question was whether to roll free shipping out to all online customers. According to the analysis, the offer increased order totals by about \$31 per order, with a plausible range of \$10 to \$52. The finance team can now settle the rollout decision with arithmetic: the lift per order, multiplied by projected online order volume, compared against the shipping cost the company absorbs. At the interval's low end the program pays for itself; at the observed lift it is clearly profitable. We recommend the rollout, with one caution. The pilot ran from March to May, so every number in this chapter describes spring ordering behavior, and the effect may be smaller in a different season. We therefore track the rollout's first quarter against the pilot's interval, revisiting the decision if the lift falls outside it.

## Exercises



### Build lab

The pilot may have changed how often customers ordered, along with how much per order. Count each pilot-eligible customer's orders in the pilot window (include zero for those who placed none: start from the snapshot's pilot column, then join counts). Compare the two groups' mean orders per customer with a permutation test and a Welch t-test, and report the difference with a bootstrap confidence interval.

### Evaluate lab

Repeat your build-lab analysis with the school-district effect in mind: rerun it excluding each group's single largest customer. State in two sentences whether the conclusion is unchanged, and why this check should be included in any analysis whose results will be shared.

> **Lab starter**
>
> A starter notebook for this lab is provided. It restates the task, reproduces the objects from this chapter that the lab builds on, and marks the cells you complete. [**Open ch-07-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-07-lab-starter.ipynb), or download it from the course page in Blackboard.